In [1]:
!pip install tensorflow[and-cuda] pandas matplotlib seaborn scikit-learn

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

2025-11-28 11:58:00.872896: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-28 11:58:01.460093: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-28 11:58:03.337600: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memory growth set for GPU")
    except RuntimeError as e:
        print(e)

Memory growth set for GPU


In [4]:
df = pd.read_csv('train.csv')

In [5]:
from tensorflow.keras.layers import TextVectorization

In [6]:
MAX_FEATURES = 200000 # number of words in the vocab

In [7]:
vectorizer = TextVectorization(max_tokens=MAX_FEATURES,
                               output_sequence_length=1800,
                               output_mode='int')

I0000 00:00:1764327499.954764    1039 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3584 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [8]:
X = df['comment_text']
y = df.drop(columns=['comment_text', 'id'])

In [9]:
vectorizer.adapt(X.values)

In [29]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [31]:
import tensorflow as tf
print(tf.test.is_gpu_available())   # legacy
print(tf.config.list_physical_devices('GPU'))  # modern

Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.


Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.


True
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


I0000 00:00:1764261469.115187     788 gpu_device.cc:2020] Created device /device:GPU:0 with 3584 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [34]:
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

RuntimeError: Physical devices cannot be modified after being initialized

In [33]:
TF_GPU_ALLOCATOR=cuda_malloc_async

NameError: name 'cuda_malloc_async' is not defined

In [10]:
vectorized_text = vectorizer(X.values)

In [11]:
dataset = tf.data.Dataset.from_tensor_slices((vectorized_text, y))
dataset = dataset.cache()
dataset = dataset.shuffle(len(X))

# SPLIT BEFORE BATCHING
train_size = int(len(X) * 0.7)
val_size = int(len(X) * 0.2)

train = dataset.take(train_size)
val = dataset.skip(train_size).take(val_size)
test = dataset.skip(train_size + val_size)

# NOW BATCH AFTER SPLITTING
train = train.batch(16).prefetch(8)
val = val.batch(16).prefetch(8)
test = test.batch(16).prefetch(8)

In [12]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Bidirectional, Dense, Embedding

In [13]:
model = Sequential()
# Create the embedding layer 
model.add(Embedding(MAX_FEATURES+1, 32))
# Bidirectional LSTM Layer
model.add(Bidirectional(LSTM(32, activation='tanh')))
# Feature extractor Fully connected layers
model.add(Dense(128, activation='relu'))
model.add(Dense(256, activation='relu'))
model.add(Dense(128, activation='relu'))
# Final layer 
model.add(Dense(6, activation='sigmoid'))

In [14]:
model.build(input_shape=(None, 1800))
model.compile(loss='BinaryCrossentropy', optimizer='Adam')
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 1800, 32)       │     6,400,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 64)             │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,491,686 (24.76 MB)

 Trainable params: 6,491,686 (24.76 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
history = model.fit(train, epochs=10, validation_data=val)

Epoch 1/10


2025-11-28 12:10:55.127761: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91600


6982/6982 ━━━━━━━━━━━━━━━━━━━━ 1420s 202ms/step - loss: 0.0631 - val_loss: 0.0503
Epoch 2/10
6982/6982 ━━━━━━━━━━━━━━━━━━━━ 1438s 206ms/step - loss: 0.0458 - val_loss: 0.0391
Epoch 3/10
6982/6982 ━━━━━━━━━━━━━━━━━━━━ 1444s 207ms/step - loss: 0.0415 - val_loss: 0.0367
Epoch 4/10
6982/6982 ━━━━━━━━━━━━━━━━━━━━ 1451s 208ms/step - loss: 0.0364 - val_loss: 0.0327
Epoch 5/10
6982/6982 ━━━━━━━━━━━━━━━━━━━━ 1455s 208ms/step - loss: 0.0333 - val_loss: 0.0279
Epoch 6/10
6982/6982 ━━━━━━━━━━━━━━━━━━━━ 1418s 203ms/step - loss: 0.0300 - val_loss: 0.0258
Epoch 7/10
6982/6982 ━━━━━━━━━━━━━━━━━━━━ 1257s 180ms/step - loss: 0.0269 - val_loss: 0.0238
Epoch 8/10
6982/6982 ━━━━━━━━━━━━━━━━━━━━ 1382s 198ms/step - loss: 0.0244 - val_loss: 0.0213
Epoch 9/10
6982/6982 ━━━━━━━━━━━━━━━━━━━━ 1414s 202ms/step - loss: 0.0220 - val_loss: 0.0188
Epoch 10/10
6982/6982 ━━━━━━━━━━━━━━━━━━━━ 1399s 200ms/step - loss: 0.0195 - val_loss: 0.0162


In [16]:
model.save('toxicity_v2.h5')

In [17]:
model.save('my_model_v2.keras')

In [18]:
model = tf.keras.models.load_model('toxicity_v2.h5')

In [19]:

from tensorflow.keras.metrics import Precision, Recall, CategoricalAccuracy

In [20]:

pre = Precision()
re = Recall()
acc = CategoricalAccuracy()

In [21]:
for X_true, y_true in test:
    yhat = model.predict(X_true)

    pre.update_state(y_true, yhat)
    re.update_state(y_true, yhat)
    acc.update_state(y_true, yhat)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 244ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 177ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/s

2025-11-28 16:10:13.979389: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [22]:
print(f'Precision: {pre.result().numpy()}, Recall:{re.result().numpy()}, Accuracy:{acc.result().numpy()}')

Precision: 0.9272019267082214, Recall:0.8981427550315857, Accuracy:0.9902243614196777


In [23]:
df_test = pd.read_csv('test.csv')
X_vec = vectorizer(df_test['comment_text'].values)

# Get predictions
preds = model.predict(X_vec) 

4787/4787 ━━━━━━━━━━━━━━━━━━━━ 358s 75ms/step


In [24]:
binary_preds = (preds > 0.5).astype(int)

In [25]:
df_final = pd.DataFrame(binary_preds,
    columns=["toxic","severe_toxic","obscene","threat","insult","identity_hate"]
)
df_final.insert(0, "id", df_test["id"])

In [26]:
df_final.to_csv("submission_v2.csv", index=False)